# Cai 2021 — NAGALA, dự báo T2D 5 năm

Đây là **same-data method reconstruction**, không phải code gốc. Notebook dùng một bản source đã khóa trong Git; không chứa pipeline riêng. CPU, seed và phương pháp được chỉ định trước. Hai mô hình: Cox sáu biến công bố và Cox sau LASSO. Random split là internal validation. Không dùng test để chọn mô hình.

Nguồn: Cai et al., 2021, DOI 10.2147/DMSO.S304994; Okamura / Dryad 10.5061/dryad.8q0p192. Xem đặc tả và giả định trong repository.

In [ ]:
import os, sys, json, hashlib, subprocess, urllib.request
from pathlib import Path
SOURCE_COMMIT = "__SOURCE_COMMIT__"
MODULE_PATH = "__MODULE_PATH__"
NOTEBOOK_REF = "__NOTEBOOK_REF__"
SOURCE_FILES = __SOURCE_FILES_JSON__
if sys.version_info < (3, 12):
    raise RuntimeError("This locked environment requires Python >=3.12")
source = Path("/tmp/cai2021-source")
source.mkdir(exist_ok=False)
for relative, expected in SOURCE_FILES.items():
    url = f"https://raw.githubusercontent.com/manhthien2005/diabetes-research/{SOURCE_COMMIT}/{MODULE_PATH}/{relative}"
    with urllib.request.urlopen(url, timeout=60) as response:
        blob = response.read()
    if hashlib.sha256(blob).hexdigest() != expected:
        raise RuntimeError("Source checksum mismatch: " + relative)
    target = source / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(blob)
(source/"source_provenance.json").write_text(json.dumps({"git_commit": SOURCE_COMMIT, "source_files": SOURCE_FILES, "notebook": NOTEBOOK_REF}), encoding="utf-8")
print("Verified committed source:", SOURCE_COMMIT)


In [ ]:
dependencies = Path("/tmp/cai2021-dependencies")
subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(dependencies), "-r", str(source/"requirements.txt")], check=True)
python = sys.executable
env = os.environ.copy()
env["PYTHONPATH"] = str(dependencies)
env.update({"OMP_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "2"})
subprocess.run([python, "-m", "unittest", "discover", "-s", str(source/"tests"), "-v"], env=env, check=True)
freeze = subprocess.check_output([python, "-m", "pip", "freeze", "--path", str(dependencies)], text=True)
Path("/kaggle/working/environment-lock.txt").write_text(freeze)


In [ ]:
workbooks = list(Path("/kaggle/input").rglob("Data set Int J Obesity.xlsx"))
if len(workbooks) != 1:
    raise RuntimeError("Expected exactly one original workbook")
config = json.loads((source/"config.json").read_text())
if hashlib.sha256(workbooks[0].read_bytes()).hexdigest() != config["dataset_sha256"]:
    raise RuntimeError("Dataset checksum differs from audited original")
env.update({"SOURCE_GIT_COMMIT": SOURCE_COMMIT, "KAGGLE_NOTEBOOK_REF": NOTEBOOK_REF, "KAGGLE_NOTEBOOK_VERSION": "1", "OMP_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "2"})
with Path("/kaggle/working/study.log").open("w") as log:
    result = subprocess.run([python, str(source/"src/run_study.py"), "--data", str(workbooks[0]), "--output", "/kaggle/working/study"], env=env, stdout=log, stderr=subprocess.STDOUT)
print(Path("/kaggle/working/study.log").read_text()[-15000:])
if result.returncode:
    raise RuntimeError("Study failed; preserve study.log and run_manifest.json")
manifest = json.loads(Path("/kaggle/working/study/run_manifest.json").read_text())
print(json.dumps({"status": manifest["status"], "split": manifest["split"], "source_sha256": manifest["source_sha256"]}, indent=2))
print(Path("/kaggle/working/study/test_results.json").read_text())
